In [0]:
from pyspark.sql import functions as F

trips = spark.table("urban_mobility.silver.trips_enriched").filter(F.col("driver_id").isNotNull())

driver_performance = (
    trips
    .groupBy("driver_id")
    .agg(
        F.count("trip_id").alias("total_trips"),
        F.sum(F.when(F.col("trip_status") == "COMPLETED", 1).otherwise(0)).alias("completed_trips"),
        F.sum(F.when(F.col("trip_status") == "CANCELLED", 1).otherwise(0)).alias("cancelled_trips"),
        F.round(F.sum(F.when(F.col("trip_status") == "COMPLETED", F.col("total_amount")).otherwise(0)), 2).alias("total_revenue"),
        F.round(F.avg(F.when(F.col("trip_status") == "COMPLETED", F.col("total_amount"))), 2).alias("avg_trip_revenue"),
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.round(F.sum(F.when(F.col("trip_status") == "COMPLETED", F.col("distance_km"))), 2).alias("total_distance_km"),
        F.sum(F.when(F.col("trip_status") == "COMPLETED", F.col("trip_duration_minutes"))).alias("total_minutes_driven"),
    )
    .withColumn(
        "completion_rate",
        F.round(F.col("completed_trips") / F.col("total_trips"), 4)
    )
    .withColumn(
        "cancellation_rate",
        F.round(F.col("cancelled_trips") / F.col("total_trips"), 4)
    )
    .withColumn(
        "revenue_per_hour",
        F.when(
            F.col("total_minutes_driven").isNotNull() & (F.col("total_minutes_driven") > 0),
            F.round(F.col("total_revenue") / (F.col("total_minutes_driven") / 60.0), 2)
        )
    )
    .drop("total_minutes_driven")
)

driver_performance.write.mode("overwrite").format("delta").saveAsTable("urban_mobility.gold.driver_performance")

result = spark.table("urban_mobility.gold.driver_performance")
print("gold.driver_performance rows:", result.count())
result.orderBy(F.desc("total_revenue")).show(5)